# 02. Preprocessing: Feature Selection and Train/Test Split

## 1. Setup and Data Loading

### 1.1 Importing Libraries

In [2]:
#necessary imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

### 1.2 Loading the Dataset

In [3]:
#loading the dataset
df = pd.read_csv('../data/zenodo_clean.csv')

In [4]:
#shape of the dataset
df.shape

(19251, 91)

## 2. Target Variable

The target is recreated here in the same way as in notebook 01: a student is labelled depressed (1) when the PHQ-9 total is 10 or more, which corresponds to moderate or worse depression. All other students are labelled 0.

### 2.1 Creating the Target Column

In [5]:
#creating the target column
df['depressed'] = (df['phq_total'] >= 10).astype(int)
df['depressed'].value_counts()

depressed
0    17967
1     1284
Name: count, dtype: int64

In [6]:
#percentage of depressed vs not depressed
df['depressed'].value_counts(normalize=True) * 100

depressed
0    93.330217
1     6.669783
Name: proportion, dtype: float64

In [7]:
#creating the three levels: 0 = okay, 1 = mild, 2 = moderate or worse
df['level'] = pd.cut(df['phq_total'], bins=[-1, 4, 9, 27], labels=[0, 1, 2]).astype(int)
df['level'].value_counts().sort_index()

level
0    11933
1     6034
2     1284
Name: count, dtype: int64

## 3. Feature Selection
### 3.1 Candidate Questions

The candidate pool contains the PHQ-2 questions, the 7 GAD-7 questions, the 7 ISI questions and the 14 PSS questions.

In [8]:
#questions we are allowed to use (phq q3-q9 are left out because they make up the answer)
phq_items = ['phq_q1', 'phq_q2']
gad_items = ['gad_q1', 'gad_q2', 'gad_q3', 'gad_q4', 'gad_q5', 'gad_q6', 'gad_q7']
isi_items = ['isi_q1', 'isi_q2', 'isi_q3', 'isi_q4', 'isi_q5', 'isi_q6', 'isi_q7']
pss_items = ['pss_q1', 'pss_q2', 'pss_q3', 'pss_q4', 'pss_q5', 'pss_q6', 'pss_q7', 'pss_q8', 'pss_q9', 'pss_q10', 'pss_q11', 'pss_q12', 'pss_q13', 'pss_q14']
items = phq_items + gad_items + isi_items + pss_items

In [9]:
#number of candidate questions
len(items)

30

#### Observation

The candidate pool has 30 questions: 2 from the PHQ-9, 7 from the GAD-7, 7 from the ISI and 14 from the PSS-14.

### 3.2 Correlation of Each Question with the PHQ-9 Total

In [10]:
#correlation of each question with the phq-9 total
corr = df[items + ['phq_total']].corr()['phq_total'].drop('phq_total').sort_values(ascending=False)
corr

phq_q2     0.723610
phq_q1     0.710337
gad_q6     0.604204
gad_q2     0.589839
gad_q3     0.586533
gad_q4     0.583413
gad_q1     0.581367
pss_q2     0.570011
pss_q3     0.562709
pss_q1     0.534699
gad_q5     0.525208
gad_q7     0.525156
isi_q1     0.493054
isi_q7     0.472410
isi_q2     0.457171
pss_q14    0.454373
isi_q4     0.448606
pss_q11    0.385191
isi_q5     0.383499
isi_q3     0.328578
pss_q8     0.326087
isi_q6     0.318061
pss_q7     0.257100
pss_q13    0.256488
pss_q9     0.232735
pss_q6     0.200826
pss_q5     0.167794
pss_q10    0.155567
pss_q4     0.113506
pss_q12    0.095765
Name: phq_total, dtype: float64

### 3.3 Selecting the Top 21 Questions

The 21 questions with the highest correlation are kept as the model features. Correlations here are calculated on all students before the split, which is a simplification. The closest call in the ranking is the one described above, and the choice is unlikely to change, since the questions at the cut-off have very similar correlations.

In [11]:
#picking the top 21 questions by correlation with phq-9
features = corr.head(21).index.tolist()
features

['phq_q2',
 'phq_q1',
 'gad_q6',
 'gad_q2',
 'gad_q3',
 'gad_q4',
 'gad_q1',
 'pss_q2',
 'pss_q3',
 'pss_q1',
 'gad_q5',
 'gad_q7',
 'isi_q1',
 'isi_q7',
 'isi_q2',
 'pss_q14',
 'isi_q4',
 'pss_q11',
 'isi_q5',
 'isi_q3',
 'pss_q8']

#### Observation

The selected features are the two PHQ-2 questions, seven GAD-7 questions, six ISI questions and six PSS questions. None of them is a PHQ-9 question from 3 to 9, so the leakage rule is respected.

## 4. Train/Test Split
### 4.1 Features and Target
`X` contains the 21 selected questions and `y` is the depression label.

In [18]:
#features (X) and target (y)
X = df[features]
y = df['level']

### 4.2 Splitting the Data

In [13]:
#splitting 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(15400, 21)
(3851, 21)


#### Observation

The training set has 15,400 students and the test set has 3,851, each with 21 features. This is an 80/20 split of the 19,251 students.

### 4.3 Checking the Class Balance

In [19]:
#checking the split kept the same share of each level
print(y_train.value_counts(normalize=True).sort_index() * 100)
print(y_test.value_counts(normalize=True).sort_index() * 100)

depressed
0    93.331169
1     6.668831
Name: proportion, dtype: float64
depressed
0    93.326409
1     6.673591
Name: proportion, dtype: float64


In [15]:
#number of depressed students in each set
print(y_train.sum())
print(y_test.sum())

1027
257


#### Observation

Stratification worked: 6.67% of the training students and 6.67% of the test students are depressed, matching the 6.67% in the full dataset. The training set contains 1,027 depressed students and the test set contains 257. Because the test set has only 257 depressed students, each one missed or caught moves recall by about 0.4 percentage points, so the test results in notebook 03 carry some uncertainty.

## 6. Saving the Split

The training and test sets are saved with their labels in the `data` folder. All models (notebooks 03 to 06) load these exact files, so any difference in results comes from the model and not from a different random split. This makes the comparison fair.

In [16]:
#saving the split so every model uses the same data
pd.concat([X_train, y_train], axis=1).to_csv('../data/train.csv', index=False)
pd.concat([X_test, y_test], axis=1).to_csv('../data/test.csv', index=False)

In [17]:
#checking the saved files
print(pd.read_csv('../data/train.csv').shape)
print(pd.read_csv('../data/test.csv').shape)

(15400, 22)
(3851, 22)


#### Observation

`train.csv` has 15,400 rows and 22 columns, and `test.csv` has 3,851 rows and 22 columns. The 22 columns are the 21 questions plus the `depressed` label. The row counts match the split in Section 4.2, so the files were written correctly.